# ЛР 02.2 — Локальный разбор ошибок и сегментов (solution)

Базовый маршрут: взять лучшую пару `model + feature_set` из ЛР 01, собрать локальные объяснения ошибок и сохранить итоговый CSV.

In [ ]:
# Что делаем: Подключаем библиотеки и настраиваем рабочее окружение ноутбука.
# Зачем: Без корректных импортов и констант следующие шаги не будут воспроизводимыми.
# Как читать результат: После выполнения этой ячейки не должно быть ошибок импорта; переменные окружения должны появиться в памяти.
# Типичные ошибки: Частая ошибка — запускать следующие ячейки до инициализации импортов и путей к данным.

# Подсказка для новичка: сначала прочитайте этот блок комментариев целиком, затем запускайте код по шагам.
# Контрольная точка: после выполнения сверяйте смысл вывода с markdown этого шага, а не только с числами.
# Если результат неожидан, остановитесь и проверьте входные данные, split и порядок запуска предыдущих ячеек.
# Подключаем зависимости для этого шага.
from pathlib import Path
import importlib.util

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd.parent,
    cwd / '02-model-interpretability-and-explainability',
    cwd.parent / '02-model-interpretability-and-explainability',
]
BASE_DIR = next((path for path in candidates if (path / 'lab_utils.py').exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError('Не удалось найти lab_utils.py. Откройте ноутбук из папки модуля 02 или из корня репозитория.')
spec = importlib.util.spec_from_file_location('lab02_utils', BASE_DIR / 'lab_utils.py')
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)

SEED = lab.SEED
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

# Подсказка для новичка: сначала прочитайте этот блок комментариев целиком, затем запускайте код по шагам.
# Контрольная точка: после выполнения сверяйте смысл вывода с markdown этого шага, а не только с числами.
# Если результат неожидан, остановитесь и проверьте входные данные, split и порядок запуска предыдущих ячеек.
# Читаем данные и артефакты, с которыми будем работать дальше.
datasets = lab.load_course_datasets()
feature_sets = lab.load_feature_sets()
model_results = lab.load_lab01_model_results()

model_factories = {
    'LogisticRegression': lambda: LogisticRegression(max_iter=3000, class_weight='balanced', random_state=SEED),
    'RandomForest': lambda: RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1,
    ),
    'LinearSVC': lambda: LinearSVC(C=1.0, class_weight='balanced', random_state=SEED),
}

error_frames = []
segment_frames = []
# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, df in datasets.items():
    x, y = lab.split_xy(df)
    x_train, x_test, y_train, y_test = lab.train_test_split_stratified(x, y)
    best_config = lab.choose_best_model_config(model_results, feature_sets, dataset_name)
    artifact = lab.fit_selected_model(
        x_train,
        x_test,
        y_train,
        y_test,
        feature_sets[dataset_name][best_config['feature_set']],
        model_factories[best_config['model']](),
    )
    error_frames.append(
        lab.build_error_case_explanations(
            artifact=artifact,
            dataset_name=dataset_name,
            model_name=best_config['model'],
            feature_set_name=best_config['feature_set'],
        )
    )
    segment_frames.append(lab.build_default_segment_tables(artifact, dataset_name))

error_case_explanations = pd.concat(error_frames, ignore_index=True)
segment_error_summary = pd.concat(segment_frames, ignore_index=True)

# Сохраняем таблицу артефакта в CSV.
error_case_explanations.to_csv(OUTPUT_DIR / 'error_case_explanations.csv', index=False)
print('Saved:', OUTPUT_DIR / 'error_case_explanations.csv')
segment_error_summary.sort_values(['dataset', 'error_rate'], ascending=[True, False]).head(10)
